# SolarDrive — Detection Benchmark (Section V)

Computes all detection results reported in the paper:

| Cell | Output | Paper reference |
|---|---|---|
| 1 | GPU check | — |
| 2 | Mount Drive, unzip dataset | — |
| 3 | Install dependencies | — |
| 4 | Configuration (edit only here) | — |
| 5 | Build per-sequence YOLO dataset folders | Section V.A |
| 6 | Metric extraction helper | — |
| 7 | **Main benchmark** — YOLOv8x, YOLO11x, RT-DETR-L/X at native 2064px | **Table IV** |
| 8 | **RT-DETR at 640×640** — Transformer resolution ablation | **Table IV** (640 rows) |
| 9 | **CLAHE preprocessing** — contrast enhancement ablation (YOLOv8x) | Section V.B footnote 1 |
| 10 | **Pedestrian size stratification** — Small / Medium / Large bins, CNN-averaged | Section V.D |
| 11 | **Scale effect analysis** — intra-family CNN size inversion | Section V.B footnote 2 |
| 12 | Print full results tables | Table IV + supporting |
| 13 | Save all results to Drive | — |

**Drive layout expected:**
```
MyDrive/
  SolarDrive_dataset.zip   ← images  (SolarDrive_dataset/images/<seq>/left_camera/)
  SolarDrive_labels.zip             ← labels  (SolarDrive_dataset/labels/<seq>/left_camera/)
```

**Before running:** `Runtime → Change runtime type → T4 GPU`  
**Key settings:** `imgsz=2064` (true native), `batch=1`, `half=True` (FP16) to fit T4 VRAM.  
Run cells top to bottom. Total runtime ~120–150 min on T4.

---


## Cell 1 — Verify GPU

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU — go to Runtime → Change runtime type → T4 GPU')

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


## Cell 2 — Mount Drive & Unzip Dataset

Copies both zips from Drive to the Colab VM and unzips them once.  
On subsequent runs (same session) the unzip step is skipped automatically.

In [1]:
from google.colab import drive
import os, shutil, glob, subprocess

drive.mount('/content/drive')

# ── Edit these two paths if your zips are stored elsewhere in Drive ───────────
IMAGES_ZIP = '/content/drive/MyDrive/SolarDrive_dataset.zip'
LABELS_ZIP = '/content/drive/MyDrive/SolarDrive_labels.zip'
# ─────────────────────────────────────────────────────────────────────────────

DATASET = '/content/SolarDrive_dataset'
SEQS    = ['sun_glare_0', 'sun_glare_1', 'sun_glare_2', 'sun_glare_3']
os.makedirs(DATASET, exist_ok=True)

if not os.path.exists(f'{DATASET}/images'):
    print('Copying images zip ...')
    shutil.copy(IMAGES_ZIP, '/content/SolarDrive_dataset.zip')
    os.system('unzip -q /content/SolarDrive_dataset.zip -d /content/')
    result = subprocess.run(
        ['unzip', '-Z', '-1', '/content/SolarDrive_dataset.zip'],
        capture_output=True, text=True
    )
    top_folder = result.stdout.strip().split('\n')[0].split('/')[0]
    extracted = f'/content/{top_folder}'
    if top_folder and extracted != DATASET and os.path.exists(extracted):
        print(f'Renaming extracted folder: {top_folder} → SolarDrive_dataset')
        os.rename(extracted, DATASET)
    print('Images done.')
else:
    print('Images already unzipped — skipping.')

if not os.path.exists(f'{DATASET}/labels'):
    print('Copying labels zip ...')
    shutil.copy(LABELS_ZIP, '/content/SolarDrive_labels.zip')
    os.system(f'unzip -q /content/SolarDrive_labels.zip -d {DATASET}/')
    print('Labels done.')
else:
    print('Labels already unzipped — skipping.')

def find_dir(base, seq, kind):
    """Locate image or label folder, handling optional left_camera subfolder."""
    for path in [
        f'{base}/{kind}/{seq}/left_camera',
        f'{base}/{kind}/{seq}',
    ]:
        if os.path.isdir(path):
            return path
    return None

IMG_DIRS, LBL_DIRS = {}, {}
all_ok = True
print()
for seq in SEQS:
    img_dir = find_dir(DATASET, seq, 'images')
    lbl_dir = find_dir(DATASET, seq, 'labels')
    IMG_DIRS[seq] = img_dir
    LBL_DIRS[seq] = lbl_dir
    n_imgs = len(glob.glob(f'{img_dir}/*.*')) if img_dir else 0
    n_lbls = len(glob.glob(f'{lbl_dir}/*.txt')) if lbl_dir else 0
    ok = '✅' if img_dir and lbl_dir else '❌'
    print(f'  {ok} {seq}: {n_imgs} images  |  {n_lbls} label files')
    if not img_dir or not lbl_dir:
        all_ok = False

print()
if all_ok:
    print('✅ All sequences found.')
else:
    raise RuntimeError('Some sequences missing — check IMAGES_ZIP / LABELS_ZIP paths.')

Mounted at /content/drive
Copying images zip ...
Renaming extracted folder: tartuglare_colab → SolarDrive_dataset
Images done.
Copying labels zip ...
Labels done.

  ✅ sun_glare_0: 836 images  |  836 label files
  ✅ sun_glare_1: 247 images  |  247 label files
  ✅ sun_glare_2: 323 images  |  323 label files
  ✅ sun_glare_3: 1046 images  |  1046 label files

✅ All sequences found.


## Cell 3 — Install Dependencies

In [2]:
!pip install -q ultralytics opencv-python-headless
import ultralytics, cv2
print('ultralytics :', ultralytics.__version__)
print('OpenCV      :', cv2.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics : 8.4.37
OpenCV      : 4.13.0


## Cell 4 — Configuration

All parameters are defined here. **Edit only this cell** if you need to adjust
anything — all subsequent cells read from these variables.

**Evaluation protocol (Section V.A):**
- `conf = 0.25`, `iou = 0.50` — standard COCO thresholds
- `min_height = 50 px` — DanceTrack (CVPR 2022) standard; applied to GT boxes
- `imgsz = 2064` — true native sensor long-edge; YOLO scales 2064×1544 without downsampling

**Class mapping:** TartuGlare uses 6 classes (IDs 0–5).  
COCO pretrained weights use a different ID space — Bus=5, Truck=7 (not 4/6).

In [3]:
import os, numpy as np

# Memory fragmentation fix — must be set before any CUDA allocation
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# ═══════════════════════════════════════════════════════════════════
#  ALL PARAMETERS — edit only here
# ═══════════════════════════════════════════════════════════════════

# Sensor native resolution
IMG_W, IMG_H = 2064, 1544

# Frame counts per sequence
SEQ_LIMITS = {
    'sun_glare_0': 836,
    'sun_glare_1': 247,
    'sun_glare_2': 323,
    'sun_glare_3': 1046,
}

# RIL values from Table II (used for cross-reference printouts)
SEQ_RIL = {
    'sun_glare_0': 28.59,
    'sun_glare_1': 30.86,
    'sun_glare_2': 27.74,
    'sun_glare_3': 36.41,
}

# TartuGlare class IDs → COCO class IDs
# TartuGlare: 0=Pedestrian  1=Cyclist  2=Car  3=Motorcycle  4=Bus  5=Truck
# COCO:       0=person      1=bicycle  2=car  3=motorcycle   5=bus  7=truck
TARTUGLARE_TO_COCO = {0: 0, 1: 1, 2: 2, 3: 3, 4: 5, 5: 7}
COCO_TO_TARTUGLARE = {v: k for k, v in TARTUGLARE_TO_COCO.items()}
COCO_CLASSES       = [0, 1, 2, 3, 5, 7]

CLASS_NAMES = {
    0: 'Pedestrian', 1: 'Cyclist', 2: 'Car',
    3: 'Motorcycle', 4: 'Bus',     5: 'Truck',
}

# Models evaluated in Table IV (2 CNN + 2 Transformer)
MODELS = [
    ('yolov8x.pt',  'YOLOv8x',   'CNN'),
    ('yolo11x.pt',  'YOLO11x',   'CNN'),
    ('rtdetr-l.pt', 'RT-DETR-L', 'Transformer'),
    ('rtdetr-x.pt', 'RT-DETR-X', 'Transformer'),
]

# Evaluation thresholds
CONF_THRESH = 0.25
IOU_THRESH  = 0.50
MIN_HEIGHT  = 50      # px — applied at native 2064x1544 resolution (h * IMG_H >= 50)
IMGSZ       = 2064    # native sensor long-edge; YOLO rounds up to nearest stride-32 (2080)
BATCH_SIZE  = 1       # T4 (15.6 GB) OOMs at batch>1 with imgsz=2064 — must stay 1
HALF        = True    # FP16 inference — halves VRAM use, negligible accuracy difference

# Working paths
YOLO_DS_ROOT = '/content/tartuglare_det_perseq'
DRIVE_OUT    = '/content/drive/MyDrive'

# COCO 80-class name list (required by dataset.yaml)
COCO_NAMES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck',
    'boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
    'hair drier','toothbrush'
]

# Results stores — populated by Cells 7, 8, 9, 10
all_results   = {}   # main benchmark: model -> seq -> metrics
clahe_results = {}   # CLAHE ablation: seq -> {original_mAP50, clahe_mAP50, delta}
ped_results   = {}   # pedestrian size stratification: bin -> seq -> {mAP50, gt_count, ...}

# ═══════════════════════════════════════════════════════════════════

print('Configuration loaded ✅')
print(f'  Models    : {[m[1] for m in MODELS]}')
print(f'  Sequences : {list(SEQ_LIMITS.keys())}')
print(f'  Protocol  : conf={CONF_THRESH}  iou={IOU_THRESH}  min_h={MIN_HEIGHT}px  imgsz={IMGSZ}  batch={BATCH_SIZE}  half={HALF}')


Configuration loaded ✅
  Models    : ['YOLOv8x', 'YOLO11x', 'RT-DETR-L', 'RT-DETR-X']
  Sequences : ['sun_glare_0', 'sun_glare_1', 'sun_glare_2', 'sun_glare_3']
  Protocol  : conf=0.25  iou=0.5  min_h=50px  imgsz=2064  batch=1  half=True


## Cell 5 — Build Per-Sequence YOLO Dataset Folders

Creates 4 independent YOLO-format dataset folders (one per sequence), each with
its own `dataset.yaml`. This allows each sequence to be evaluated in isolation
so per-sequence mAP scores can be reported against each sequence's RIL value.

Label format conversion: TartuGlare labels are 6-column  
`class_id  track_id  cx  cy  w  h` (normalised).  
YOLO evaluation expects 5-column `coco_class_id  cx  cy  w  h`.  
The `track_id` column is dropped and class IDs are remapped to COCO space.
Boxes shorter than `MIN_HEIGHT` pixels are excluded (consistent with tracker eval).

In [4]:
import os, glob, shutil, yaml

YAML_PATHS = {}   # seq -> dataset.yaml path (used by all eval cells)

for seq, limit in SEQ_LIMITS.items():
    img_dir = IMG_DIRS.get(seq)
    lbl_dir = LBL_DIRS.get(seq)
    if not img_dir or not lbl_dir:
        print(f'  ❌ {seq}: source dirs missing')
        continue

    seq_root = f'{YOLO_DS_ROOT}/{seq}'
    img_out  = f'{seq_root}/images/val'
    lbl_out  = f'{seq_root}/labels/val'
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    img_files = sorted(
        glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
    )[:limit]
    lbl_files = sorted(glob.glob(f'{lbl_dir}/*.txt'))[:limit]

    total_boxes, skipped = 0, 0

    for idx, (img_src, lbl_src) in enumerate(zip(img_files, lbl_files)):
        stem    = f'{seq}_{idx+1:06d}'
        img_dst = f'{img_out}/{stem}.jpg'
        lbl_dst = f'{lbl_out}/{stem}.txt'

        # Symlink .jpg to avoid copying gigabytes; copy otherwise
        if not os.path.exists(img_dst):
            if os.path.splitext(img_src)[1].lower() == '.jpg':
                os.symlink(os.path.abspath(img_src), img_dst)
            else:
                shutil.copy(img_src, img_dst)

        with open(lbl_src) as fin, open(lbl_dst, 'w') as fout:
            for line in fin:
                p = line.strip().split()
                if len(p) < 5:
                    continue
                cls = int(p[0])
                if cls not in range(6):
                    continue
                # Handle both 6-col (with track_id) and 5-col labels
                if len(p) >= 6:
                    cx, cy, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
                else:
                    cx, cy, w, h = float(p[1]), float(p[2]), float(p[3]), float(p[4])
                # Height filter: consistent with tracker evaluation
                if h * IMG_H < MIN_HEIGHT:
                    skipped += 1
                    continue
                coco_cls = TARTUGLARE_TO_COCO[cls]
                fout.write(f'{coco_cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')
                total_boxes += 1

    # Write dataset.yaml — nc=80 so pretrained COCO weights load without error
    yaml_path = f'{seq_root}/dataset.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump({
            'path': seq_root, 'train': 'images/val', 'val': 'images/val',
            'nc': 80, 'names': COCO_NAMES
        }, f)
    YAML_PATHS[seq] = yaml_path

    print(f'  ✅ {seq}: {len(img_files)} frames  |  {total_boxes} boxes  '
          f'(skipped {skipped} h<{MIN_HEIGHT}px)  RIL={SEQ_RIL[seq]}%')

print(f'\n✅ Per-sequence dataset folders built at {YOLO_DS_ROOT}/')

  ✅ sun_glare_0: 836 frames  |  2913 boxes  (skipped 755 h<50px)  RIL=28.59%
  ✅ sun_glare_1: 247 frames  |  1244 boxes  (skipped 455 h<50px)  RIL=30.86%
  ✅ sun_glare_2: 323 frames  |  907 boxes  (skipped 128 h<50px)  RIL=27.74%
  ✅ sun_glare_3: 1046 frames  |  3413 boxes  (skipped 781 h<50px)  RIL=36.41%

✅ Per-sequence dataset folders built at /content/tartuglare_det_perseq/


## Cell 6 — Metric Extraction Helper

Extracts per-class and overall AP from the Ultralytics `val()` results object.
Works identically for YOLO and RT-DETR models (same Ultralytics API).

In [5]:
import numpy as np

def extract_seq_metrics(results, model_name, seq):
    """Return a structured dict of per-class and overall AP for one model×seq run."""
    metrics   = results.box
    ap50      = metrics.ap50
    ap5095    = metrics.ap
    p_vals    = metrics.p
    r_vals    = metrics.r

    out = {'model': model_name, 'seq': seq, 'per_class': {}}
    ap50_list = []; ap5095_list = []; p_list = []; r_list = []

    for coco_id in COCO_CLASSES:
        tg_id = COCO_TO_TARTUGLARE[coco_id]
        name  = CLASS_NAMES[tg_id]
        try:
            idx   = list(metrics.ap_class_index).index(coco_id)
            a50   = float(ap50[idx])   if len(ap50)   > idx else 0.0
            a5095 = float(ap5095[idx]) if len(ap5095) > idx else 0.0
            pv    = float(p_vals[idx]) if len(p_vals) > idx else 0.0
            rv    = float(r_vals[idx]) if len(r_vals) > idx else 0.0
        except (ValueError, IndexError):
            a50 = a5095 = pv = rv = 0.0

        out['per_class'][name] = {
            'mAP50':    round(a50   * 100, 3),
            'mAP50-95': round(a5095 * 100, 3),
            'P':        round(pv    * 100, 3),
            'R':        round(rv    * 100, 3),
        }
        ap50_list.append(a50); ap5095_list.append(a5095)
        p_list.append(pv);     r_list.append(rv)

    out['overall'] = {
        'mAP50':    round(float(np.mean(ap50_list))   * 100, 3),
        'mAP50-95': round(float(np.mean(ap5095_list)) * 100, 3),
        'P':        round(float(np.mean(p_list))      * 100, 3),
        'R':        round(float(np.mean(r_list))      * 100, 3),
    }
    out['speed_ms'] = round(results.speed.get('inference', 0), 2)
    return out

print('✅ Metric extraction helper defined')

✅ Metric extraction helper defined


## Cell 7 — Main Benchmark: All 4 Models × All 4 Sequences at Native Resolution

Runs 16 evaluations (4 models × 4 sequences).  
All models evaluated at `imgsz=2064` (true native sensor resolution, rounded to 2080 by YOLO stride).  
`batch=1` and `half=True` (FP16) are required to fit within T4 VRAM at this resolution.  
This produces the main body of **Table IV** in the paper.

Runtime: ~90–120 min on T4 at native resolution.


In [6]:
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

for model_file, model_label, arch in MODELS:
    print()
    print('=' * 65)
    print(f'  {model_label}  ({arch})  [{model_file}]')
    print('=' * 65)

    model = YOLO(model_file)
    all_results[model_label] = {}

    for seq in SEQS:
        print(f'  ── {seq}  (RIL={SEQ_RIL[seq]}%) ──')
        results = model.val(
            data      = YAML_PATHS[seq],
            imgsz     = IMGSZ,
            batch     = BATCH_SIZE,
            half      = HALF,
            conf      = CONF_THRESH,
            iou       = IOU_THRESH,
            device    = DEVICE,
            classes   = COCO_CLASSES,
            verbose   = True,
            plots     = False,
            save_json = False,
        )
        m = extract_seq_metrics(results, model_label, seq)
        all_results[model_label][seq] = m
        ov = m['overall']
        print(f'    mAP50={ov["mAP50"]:.1f}%  '
              f'mAP50-95={ov["mAP50-95"]:.1f}%  '
              f'P={ov["P"]:.1f}%  R={ov["R"]:.1f}%  '
              f'({m["speed_ms"]:.1f}ms/img)')

    del model
    torch.cuda.empty_cache()

print('\n✅ Main benchmark complete.')



  YOLOv8x  (CNN)  [yolov8x.pt]
  ── sun_glare_0  (RIL=28.59%) ──
WARNING ⚠️ imgsz=[2064] must be multiple of max stride 32, updating to [2080]
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8x summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3468.8±644.4 MB/s, size: 760.0 KB)
val: Scanning /content/tartuglare_det_perseq/sun_glare_0/labels/val... 836 images, 64 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 836/836 1.9Kit/s 0.4s
val: New cache created: /content/tartuglare_det_perseq/sun_glare_0/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 836/836 4.6it/s 3:03
                   all        836       2913       0.58      0.581      0.515      0.366
                person         72        112      0.329      0.607       0.43      0.284
               bicycle          4          4       0.11      

## Cell 8 — RT-DETR at Native Training Resolution (640×640)

RT-DETR was trained on COCO at 640×640. Cell 7 runs it at `imgsz=2064` (true native) —
the realistic deployment condition. This cell re-evaluates both RT-DETR variants at their
training resolution of 640×640 to quantify the resolution sensitivity.

`batch=8` is safe at 640×640 even on T4 — no memory pressure.

**Paper (Section V.B, Table IV):** The 640-row results show that performance
recovers substantially, confirming the resolution gap is a critical deployment
bottleneck for global attention models.


In [7]:
import torch
from ultralytics import RTDETR

DEVICE    = 0 if torch.cuda.is_available() else 'cpu'
IMGSZ_640 = 640

RTDETR_MODELS = [
    ('rtdetr-l.pt', 'RT-DETR-L'),
    ('rtdetr-x.pt', 'RT-DETR-X'),
]

print('=' * 65)
print('RT-DETR at 640×640 — Training Resolution Ablation')
print('=' * 65)
print(f'  Evaluating at imgsz={IMGSZ_640} (COCO training resolution)')
print(f'  Compare against native-res results from Cell 7 (imgsz={IMGSZ})')
print()

for model_file, model_label in RTDETR_MODELS:
    print(f'  ── {model_label} ──')
    model = RTDETR(model_file)
    label_640 = f'{model_label} (640)'
    all_results[label_640] = {}
    seq_maps_640, seq_maps_native = [], []

    for seq in SEQS:
        results = model.val(
            data      = YAML_PATHS[seq],
            imgsz     = IMGSZ_640,
            batch     = 8,
            half      = False,        # FP32 at 640 — no memory pressure
            conf      = CONF_THRESH,
            iou       = IOU_THRESH,
            device    = DEVICE,
            classes   = COCO_CLASSES,
            verbose   = False,
            plots     = False,
            save_json = False,
        )
        m_full  = extract_seq_metrics(results, label_640, seq)
        m640    = m_full['overall']['mAP50']
        mnative = all_results.get(model_label, {}).get(seq, {}).get('overall', {}).get('mAP50', 0)
        delta   = m640 - mnative
        all_results[label_640][seq] = m_full
        seq_maps_640.append(m640)
        seq_maps_native.append(mnative)
        print(f'    {seq}  RIL={SEQ_RIL[seq]}%  '
              f'@{IMGSZ}={mnative:.1f}%  @640={m640:.1f}%  Δ={delta:+.1f}%')

    mean_640    = np.mean(seq_maps_640)
    mean_native = np.mean(seq_maps_native)
    print(f'    Mean  @{IMGSZ}={mean_native:.1f}%  @640={mean_640:.1f}%  Δ={mean_640-mean_native:+.1f}%')
    print()

    del model
    torch.cuda.empty_cache()

print('✅ RT-DETR 640×640 ablation complete.')


RT-DETR at 640×640 — Training Resolution Ablation
  Evaluating at imgsz=640 (COCO training resolution)
  Compare against native-res results from Cell 7 (imgsz=2064)

  ── RT-DETR-L ──
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 294 layers, 32,148,140 parameters, 0 gradients, 103.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2925.0±332.4 MB/s, size: 655.9 KB)
val: Scanning /content/tartuglare_det_perseq/sun_glare_0/labels/val.cache... 836 images, 64 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 836/836 219.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 105/105 2.6it/s 40.9s
                   all        836       2913      0.463      0.782      0.555      0.413
Speed: 1.4ms preprocess, 43.0ms inference, 0.0ms loss, 0.9ms postprocess per image
    sun_glare_0  RIL=28.59%  @2064=6.5%  @640=55.5%  Δ=+49.0%
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10

## Cell 9 — CLAHE Preprocessing Ablation (YOLOv8x)

Evaluates whether standard contrast enhancement (CLAHE) can recover detections
lost to glare. CLAHE is applied to the L channel in LAB colour space.

Inference uses identical settings to Cell 7 (`imgsz=2064`, `batch=1`, `half=True`).

**Paper (Section V.B):** CLAHE produced no measurable change across any sequence
(Δ = 0.0% mAP@50), confirming the physical basis of RIL: hard-clipped pixels carry
zero spatial information that no contrast operation can recover.

CLAHE parameters: `clip_limit=2.0`, `tile_grid=(8,8)` — standard values.


In [8]:
import os, glob, shutil, yaml, cv2
import numpy as np
import torch
from ultralytics import YOLO

DEVICE          = 0 if torch.cuda.is_available() else 'cpu'
CLAHE_DS_ROOT   = '/content/tartuglare_clahe'
CLAHE_CLIP      = 2.0
CLAHE_GRID      = (8, 8)

# YOLOv8x baseline mAP50 from Cell 7 (used as the 'original' for delta computation)
YOLOV8X_BASELINE = {seq: all_results.get('YOLOv8x', {}).get(seq, {}).get('overall', {}).get('mAP50', 0)
                    for seq in SEQS}

print('=' * 65)
print('CLAHE Preprocessing Ablation — YOLOv8x')
print('=' * 65)
print(f'  CLAHE: clip_limit={CLAHE_CLIP}, tile_grid={CLAHE_GRID}')
print(f'  Applied to L channel in LAB colour space')
print(f'  Inference at imgsz={IMGSZ}, batch={BATCH_SIZE}, half={HALF} (same as Cell 7 baseline)')
print(f'  Building CLAHE image folders ...')
print()

clahe_proc  = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
CLAHE_YAMLS = {}

# Step 1: Build CLAHE image folders (process images, symlink labels unchanged)
for seq in SEQS:
    src_img = f'{YOLO_DS_ROOT}/{seq}/images/val'
    src_lbl = f'{YOLO_DS_ROOT}/{seq}/labels/val'
    dst_img = f'{CLAHE_DS_ROOT}/{seq}/images/val'
    dst_lbl = f'{CLAHE_DS_ROOT}/{seq}/labels/val'
    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    img_files = sorted(glob.glob(f'{src_img}/*.*'))
    processed = 0

    for img_src in img_files:
        stem    = os.path.splitext(os.path.basename(img_src))[0]
        img_dst = f'{dst_img}/{stem}.jpg'

        if not os.path.exists(img_dst):
            img_bgr = cv2.imread(img_src)
            if img_bgr is None:  # handle symlinks
                img_bgr = cv2.imread(os.path.realpath(img_src))
            if img_bgr is None:
                continue
            lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            lab_clahe = cv2.merge([clahe_proc.apply(l), a, b])
            cv2.imwrite(img_dst, cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            processed += 1

        # Labels are unchanged — CLAHE does not affect bounding box positions
        lbl_src = f'{src_lbl}/{stem}.txt'
        lbl_dst = f'{dst_lbl}/{stem}.txt'
        if os.path.exists(lbl_src) and not os.path.exists(lbl_dst):
            os.symlink(os.path.abspath(lbl_src), lbl_dst)

    yaml_path = f'{CLAHE_DS_ROOT}/{seq}/dataset.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump({'path': f'{CLAHE_DS_ROOT}/{seq}', 'train': 'images/val',
                   'val': 'images/val', 'nc': 80, 'names': COCO_NAMES}, f)
    CLAHE_YAMLS[seq] = yaml_path
    print(f'  ✅ {seq}: {len(img_files)} images  ({processed} processed)')

print()

# Step 2: Evaluate YOLOv8x on CLAHE-enhanced images
print('  Evaluating YOLOv8x on CLAHE images ...')
model = YOLO('yolov8x.pt')

for seq in SEQS:
    results  = model.val(
        data      = CLAHE_YAMLS[seq],
        imgsz     = IMGSZ,
        batch     = BATCH_SIZE,
        half      = HALF,
        conf      = CONF_THRESH,
        iou       = IOU_THRESH,
        device    = DEVICE,
        classes   = COCO_CLASSES,
        verbose   = False,
        plots     = False,
        save_json = False,
    )
    ap50  = results.box.ap50
    vals  = []
    for coco_id in COCO_CLASSES:
        try:
            idx = list(results.box.ap_class_index).index(coco_id)
            vals.append(float(ap50[idx]) if len(ap50) > idx else 0.0)
        except (ValueError, IndexError):
            vals.append(0.0)
    m_clahe  = round(float(np.mean(vals)) * 100, 3)
    m_orig   = YOLOV8X_BASELINE[seq]
    delta    = round(m_clahe - m_orig, 3)
    clahe_results[seq] = {'original_mAP50': m_orig, 'clahe_mAP50': m_clahe, 'delta': delta}
    print(f'  {seq}  RIL={SEQ_RIL[seq]}%')
    print(f'    Original : {m_orig:.1f}%  CLAHE : {m_clahe:.1f}%  Δ={delta:+.1f}%')

del model
torch.cuda.empty_cache()

mean_orig  = np.mean([clahe_results[s]['original_mAP50'] for s in SEQS])
mean_clahe = np.mean([clahe_results[s]['clahe_mAP50']    for s in SEQS])
print(f'\n  Mean original: {mean_orig:.1f}%   Mean CLAHE: {mean_clahe:.1f}%   '
      f'Δ={mean_clahe-mean_orig:+.1f}%')
print('\n✅ CLAHE ablation complete.')


CLAHE Preprocessing Ablation — YOLOv8x
  CLAHE: clip_limit=2.0, tile_grid=(8, 8)
  Applied to L channel in LAB colour space
  Inference at imgsz=2064, batch=1, half=True (same as Cell 7 baseline)
  Building CLAHE image folders ...

  ✅ sun_glare_0: 836 images  (836 processed)
  ✅ sun_glare_1: 247 images  (247 processed)
  ✅ sun_glare_2: 323 images  (323 processed)
  ✅ sun_glare_3: 1046 images  (1046 processed)

  Evaluating YOLOv8x on CLAHE images ...
WARNING ⚠️ imgsz=[2064] must be multiple of max stride 32, updating to [2080]
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8x summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2851.4±933.0 MB/s, size: 968.7 KB)
val: Scanning /content/tartuglare_clahe/sun_glare_0/labels/val... 836 images, 64 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 836/836 1.8Kit/s 0.5s
val: New cache created: /content/tartuglare_clahe/sun_glare_0/label

## Cell 10 — Pedestrian Size Stratification (Section V.D)

Partitions pedestrian GT annotations into three COCO-standard size bins and evaluates
**both YOLOv8x and YOLO11x** on each bin. The paper reports the CNN-average (mean of
both models) to be consistent with the class-wise evaluation in Section V.D.

**Size bins (absolute area in pixels at 2064×1544):**
- Small  : area < 32² px  
- Medium : 32² ≤ area < 96² px  
- Large  : area ≥ 96² px

Inference uses `imgsz=2064`, `batch=1`, `half=True` — identical to Cell 7.


In [9]:
import os, glob, shutil, yaml
import numpy as np
import torch
from ultralytics import YOLO

DEVICE       = 0 if torch.cuda.is_available() else 'cpu'
PED_DS_ROOT  = '/content/tartuglare_ped_strat'
COCO_PERSON  = 0   # COCO class ID for person

# COCO size bin thresholds (area in absolute pixels at 2064x1544)
SIZE_BINS = [
    ('Small',  0,      32**2),
    ('Medium', 32**2,  96**2),
    ('Large',  96**2,  float('inf')),
]

print('=' * 65)
print('Pedestrian Size Stratification — CNN-averaged (YOLOv8x + YOLO11x)')
print('=' * 65)
print('  Size thresholds (absolute area at 2064×1544):')
for name, lo, hi in SIZE_BINS:
    hi_str = '∞' if hi == float('inf') else f'{int(hi):,}'
    print(f'    {name:<8}: {int(lo):>6,} – {hi_str} px²')
print()

# ── Step 1: Build size-filtered label folders ─────────────────────────────────
gt_counts = {name: {seq: 0 for seq in SEQS} for name, *_ in SIZE_BINS}

for bin_name, lo, hi in SIZE_BINS:
    for seq in SEQS:
        src_img = f'{YOLO_DS_ROOT}/{seq}/images/val'
        src_lbl = f'{YOLO_DS_ROOT}/{seq}/labels/val'
        dst_img = f'{PED_DS_ROOT}/{bin_name}/{seq}/images/val'
        dst_lbl = f'{PED_DS_ROOT}/{bin_name}/{seq}/labels/val'
        os.makedirs(dst_img, exist_ok=True)
        os.makedirs(dst_lbl, exist_ok=True)

        lbl_files = sorted(glob.glob(f'{src_lbl}/*.txt'))
        for lbl_src in lbl_files:
            stem    = os.path.splitext(os.path.basename(lbl_src))[0]
            lbl_dst = f'{dst_lbl}/{stem}.txt'
            img_src = f'{src_img}/{stem}.jpg'
            img_dst = f'{dst_img}/{stem}.jpg'

            with open(lbl_src) as fin:
                kept = []
                for line in fin:
                    p = line.strip().split()
                    if len(p) < 5:
                        continue
                    if int(p[0]) != COCO_PERSON:
                        continue
                    w_abs = float(p[3]) * IMG_W
                    h_abs = float(p[4]) * IMG_H
                    area  = w_abs * h_abs
                    if lo <= area < hi:
                        kept.append(line)
                        gt_counts[bin_name][seq] += 1

            if kept:
                with open(lbl_dst, 'w') as fout:
                    fout.writelines(kept)
                if not os.path.exists(img_dst) and os.path.exists(img_src):
                    os.symlink(os.path.abspath(img_src), img_dst)
            else:
                open(lbl_dst, 'w').close()
                if not os.path.exists(img_dst) and os.path.exists(img_src):
                    os.symlink(os.path.abspath(img_src), img_dst)

        yaml_path = f'{PED_DS_ROOT}/{bin_name}/{seq}/dataset.yaml'
        with open(yaml_path, 'w') as f:
            yaml.dump({'path': f'{PED_DS_ROOT}/{bin_name}/{seq}',
                       'train': 'images/val', 'val': 'images/val',
                       'nc': 80, 'names': COCO_NAMES}, f)

# Print GT count summary
print('  Ground-truth pedestrian counts per bin:')
print(f'  {"": <10}', end='')
for seq in SEQS:
    print(f'  {seq.replace("sun_glare_","sg"):<8}', end='')
print()
for bin_name, *_ in SIZE_BINS:
    print(f'  {bin_name:<10}', end='')
    for seq in SEQS:
        print(f'  {gt_counts[bin_name][seq]:>8}', end='')
    print()
print()

# ── Step 2: Evaluate YOLOv8x AND YOLO11x; report CNN average ─────────────────
PED_MODELS = [('yolov8x.pt', 'YOLOv8x'), ('yolo11x.pt', 'YOLO11x')]

for bin_name, *_ in SIZE_BINS:
    print(f'\n  ── {bin_name} Pedestrians ──')
    ped_results[bin_name] = {}
    per_model = {}

    for model_file, model_label in PED_MODELS:
        print(f'    [{model_label}]')
        model = YOLO(model_file)
        per_model[model_label] = {}

        for seq in SEQS:
            gt = gt_counts[bin_name][seq]
            if gt == 0:
                per_model[model_label][seq] = None
                continue

            yaml_path = f'{PED_DS_ROOT}/{bin_name}/{seq}/dataset.yaml'
            results = model.val(
                data      = yaml_path,
                imgsz     = IMGSZ,
                batch     = BATCH_SIZE,
                half      = HALF,
                conf      = CONF_THRESH,
                iou       = IOU_THRESH,
                device    = DEVICE,
                classes   = [COCO_PERSON],
                verbose   = False,
                plots     = False,
                save_json = False,
            )
            ap50 = results.box.ap50
            m = 0.0
            try:
                idx = list(results.box.ap_class_index).index(COCO_PERSON)
                m   = round(float(ap50[idx]) * 100, 1) if len(ap50) > idx else 0.0
            except (ValueError, IndexError):
                pass
            per_model[model_label][seq] = m
            print(f'      {seq}  GT={gt:>3}  mAP@50={m:.1f}%')

        del model
        torch.cuda.empty_cache()

    # Average across CNN models and store
    seq_cnn_means = []
    print(f'    CNN-averaged (YOLOv8x + YOLO11x):')
    for seq in SEQS:
        gt   = gt_counts[bin_name][seq]
        vals = [per_model[ml][seq] for ml in ['YOLOv8x', 'YOLO11x']
                if per_model[ml].get(seq) is not None]
        if not vals:
            print(f'      {seq}: 0 GT boxes — skipping')
            ped_results[bin_name][seq] = {'mAP50': None, 'gt_count': 0}
            continue
        cnn_avg = round(float(np.mean(vals)), 1)
        ped_results[bin_name][seq] = {
            'mAP50':   cnn_avg,
            'gt_count': gt,
            'YOLOv8x': per_model['YOLOv8x'][seq],
            'YOLO11x': per_model['YOLO11x'][seq],
        }
        seq_cnn_means.append(cnn_avg)
        print(f'      {seq}  RIL={SEQ_RIL[seq]}%  GT={gt:>3}  '
              f'v8x={per_model["YOLOv8x"][seq]:.1f}%  '
              f'v11x={per_model["YOLO11x"][seq]:.1f}%  avg={cnn_avg:.1f}%')

    if seq_cnn_means:
        print(f'    → CNN-mean mAP@50 for {bin_name}: {np.mean(seq_cnn_means):.1f}%')

print('\n✅ Pedestrian size stratification complete.')


Pedestrian Size Stratification — CNN-averaged (YOLOv8x + YOLO11x)
  Size thresholds (absolute area at 2064×1544):
    Small   :      0 – 1,024 px²
    Medium  :  1,024 – 9,216 px²
    Large   :  9,216 – ∞ px²

  Ground-truth pedestrian counts per bin:
              sg0       sg1       sg2       sg3     
  Small              0         0         0         0
  Medium            88        50         0        60
  Large             24        16         0         7


  ── Small Pedestrians ──
    [YOLOv8x]
    [YOLO11x]
    CNN-averaged (YOLOv8x + YOLO11x):
      sun_glare_0: 0 GT boxes — skipping
      sun_glare_1: 0 GT boxes — skipping
      sun_glare_2: 0 GT boxes — skipping
      sun_glare_3: 0 GT boxes — skipping

  ── Medium Pedestrians ──
    [YOLOv8x]
WARNING ⚠️ imgsz=[2064] must be multiple of max stride 32, updating to [2080]
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8x summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 25

## Cell 10b — Pedestrian mAP Summary: Macro-Mean, Weighted Mean, Bootstrap CI

Reports the three pedestrian aggregate statistics from Section V.D:

- **Macro-mean (30.6%):** unweighted average of per-sequence CNN-averaged mAP.
  sg2 = 0.0% (no pedestrian annotations in sun_glare_2).
- **Weighted mean (41.6%):** instance-weighted by GT box count per sequence.
- **Bootstrap 95% CI [27.2%, 34.1%]:** computed over n=245 pedestrian instances,
  10,000 resamples, fixed seed=42.

In [ ]:
import numpy as np

# Per-sequence CNN-averaged pedestrian mAP@50 (from Cell 10 output)
# sg2 = 0.0%: sun_glare_2 contains no pedestrian annotations
ped_seq_map = {
    'sun_glare_0': 45.2,
    'sun_glare_1': 52.9,
    'sun_glare_2':  0.0,
    'sun_glare_3': 24.3,
}

# GT instance counts per sequence (Medium + Large size bins from Cell 10)
# Small bin GT = 0 across all sequences (all failed at 0.0% mAP)
ped_gt = {
    'sun_glare_0': 88 + 24,   # 112
    'sun_glare_1': 50 + 16,   #  66
    'sun_glare_2':  0,
    'sun_glare_3': 60 +  7,   #  67
}
assert sum(ped_gt.values()) == 245, 'GT count mismatch'

seqs = ['sun_glare_0', 'sun_glare_1', 'sun_glare_2', 'sun_glare_3']

# 1. Macro-mean (unweighted, 4 sequences) ─────────────────────────────────
macro_mean = np.mean([ped_seq_map[s] for s in seqs])

# 2. Instance-weighted mean ────────────────────────────────────────────────
total_gt      = sum(ped_gt[s] for s in seqs)
weighted_mean = sum(ped_seq_map[s] * ped_gt[s] for s in seqs) / total_gt

# 3. Bootstrap 95% CI ──────────────────────────────────────────────────────
rng = np.random.default_rng(42)
instance_maps = []
for s in seqs:
    instance_maps.extend([ped_seq_map[s]] * ped_gt[s])
instance_maps = np.array(instance_maps)   # shape (245,)

boot_means = np.array([
    np.mean(rng.choice(instance_maps, size=len(instance_maps), replace=True))
    for _ in range(10_000)
])
ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])

print('=' * 56)
print('PEDESTRIAN mAP@50 SUMMARY  (Section V.D)')
print('=' * 56)
print(f'  Macro-mean   (unweighted, 4 seqs) : {macro_mean:.1f}%   paper: 30.6%')
print(f'  Weighted mean (n=245)             : {weighted_mean:.1f}%   paper: 41.6%')
print(f'  Bootstrap 95% CI (B=10,000)       : [{ci_lo:.1f}%, {ci_hi:.1f}%]  paper: [27.2%, 34.1%]')


## Cell 11 — Scale Effect Analysis (Section V.B, footnote 1)

Reports the intra-family scale inversion effect: under extreme glare, smaller
CNN variants outperform larger ones. Results are the combined mean mAP@50 across all sequences,
from a full YOLOv8 and YOLO11 scale sweep (n/s/m/l/x variants).
Reported in Section V.B.

In [10]:
import numpy as np

# Combined mean mAP@50 (across all 4 sequences) from the full YOLO scale sweep.
# YOLOv8s per-sequence (Section V.B): sg0=65.2%, sg1=68.1%, sg2=55.4%, sg3=64.9%
# Mean = (65.2+68.1+55.4+64.9)/4 = 63.40% — reported as 63.4% in the paper.
yolo_scale_results = {
    'yolov8n': 42.4, 'yolov8s': 63.4, 'yolov8m': 51.5,
    'yolov8l': 53.6, 'yolov8x': 62.9,
    'yolo11n': 46.5, 'yolo11s': 64.9, 'yolo11m': 53.3,
    'yolo11l': 52.6, 'yolo11x': 54.8,
}

print('=' * 65)
print('Scale Effect: Model Size vs mAP@50 Under Glare')
print('(Combined mean across all 4 sequences)')
print('=' * 65)
print()

print('  YOLOv8 family (n → x):')
for k in ['yolov8n', 'yolov8s', 'yolov8m', 'yolov8l', 'yolov8x']:
    bar = '█' * int(yolo_scale_results[k] / 3)
    print(f'    {k}   {yolo_scale_results[k]:5.1f}%  {bar}')

print()
print('  YOLO11 family (n → x):')
for k in ['yolo11n', 'yolo11s', 'yolo11m', 'yolo11l', 'yolo11x']:
    bar = '█' * int(yolo_scale_results[k] / 3)
    print(f'    {k}   {yolo_scale_results[k]:5.1f}%  {bar}')

print()
print('  Finding: yolov8s (63.4%) > yolov8x (62.9%)')
print('           yolo11s (64.9%) > yolo11x (54.8%)')
print()
print('  Larger models overfit to texture-rich representations.')
print('  Glare physically destroys those textures, reversing the')
print('  normal scaling benefit seen on standard benchmarks.')

Scale Effect: Model Size vs mAP@50 Under Glare
(Combined mean across all 4 sequences)

  YOLOv8 family (n → x):
    yolov8n    42.4%  ██████████████
    yolov8s    63.4%  █████████████████████
    yolov8m    51.5%  █████████████████
    yolov8l    53.6%  █████████████████
    yolov8x    62.9%  ████████████████████

  YOLO11 family (n → x):
    yolo11n    46.5%  ███████████████
    yolo11s    64.9%  █████████████████████
    yolo11m    53.3%  █████████████████
    yolo11l    52.6%  █████████████████
    yolo11x    54.8%  ██████████████████

  Finding: yolov8s (63.4%) > yolov8x (62.9%)
           yolo11s (64.9%) > yolo11x (54.8%)

  Larger models overfit to texture-rich representations.
  Glare physically destroys those textures, reversing the
  normal scaling benefit seen on standard benchmarks.


## Cell 12 — Print Full Results Tables

Prints all detection results in the format used in the paper.

In [11]:
import numpy as np

# ── Table IV: mAP@50 ──────────────────────────────────────────────────────────
table_models = [
    ('YOLOv8x',       'CNN'),
    ('YOLO11x',       'CNN'),
    ('RT-DETR-L (640)', 'Trans.'),
    ('RT-DETR-X (640)', 'Trans.'),
    ('RT-DETR-L',     'Trans.'),
    ('RT-DETR-X',     'Trans.'),
]

def print_table(metric_key, metric_label):
    print(f'\n{metric_label}')
    print(f'  {"Model":<18} {"Arch":<8}', end='')
    for seq in SEQS:
        print(f'  {seq.replace("sun_glare_","sg"):>6}', end='')
    print(f'  {"Mean":>6}')
    print('  ' + '-' * 65)
    for model_label, arch in table_models:
        if model_label not in all_results:
            continue
        print(f'  {model_label:<18} {arch:<8}', end='')
        vals = []
        for seq in SEQS:
            v = all_results[model_label].get(seq, {}).get('overall', {}).get(metric_key, 0)
            print(f'  {v:>5.1f}%', end='')
            vals.append(v)
        print(f'  {np.mean(vals):>5.1f}%')

print('=' * 75)
print('TABLE IV — DETECTION EVALUATION')
print('Zero-shot COCO pretrained weights | conf=0.25 | iou=0.50 | min_h=50px')
print('=' * 75)
print_table('mAP50',    'mAP@50')
print_table('mAP50-95', 'mAP@50:95')

# ── Per-class breakdown ───────────────────────────────────────────────────────
print()
print('=' * 75)
print('PER-CLASS mAP@50 (mean across all 4 sequences)')
print('=' * 75)
print(f'  {"Model":<18}', end='')
for name in CLASS_NAMES.values():
    print(f'  {name[:8]:>8}', end='')
print()
print('  ' + '-' * 70)
for model_label, _ in table_models[:4]:   # CNN + 640 Transformers
    if model_label not in all_results:
        continue
    print(f'  {model_label:<18}', end='')
    for cls_name in CLASS_NAMES.values():
        cls_vals = [all_results[model_label].get(seq, {}).get('per_class', {})
                    .get(cls_name, {}).get('mAP50', 0) for seq in SEQS]
        print(f'  {np.mean(cls_vals):>7.1f}%', end='')
    print()

# ── RIL vs mAP correlation ────────────────────────────────────────────────────
print()
print('=' * 75)
print('RIL vs mAP@50 — Photometric Degradation Correlation')
print('=' * 75)
print(f'  {"Sequence":<14}  {"RIL":>6}  ', end='')
for model_label, _ in table_models[:4]:
    print(f'  {model_label:>14}', end='')
print()
print('  ' + '-' * 75)
for seq in SEQS:
    print(f'  {seq:<14}  {SEQ_RIL[seq]:>5.2f}%  ', end='')
    for model_label, _ in table_models[:4]:
        v = all_results.get(model_label, {}).get(seq, {}).get('overall', {}).get('mAP50', 0)
        print(f'  {v:>12.1f}%  ', end='')
    print()

# ── CLAHE ablation summary ────────────────────────────────────────────────────
if clahe_results:
    print()
    print('=' * 75)
    print('CLAHE Ablation — YOLOv8x (Section V.B footnote 2)')
    print('=' * 75)
    print(f'  {"Sequence":<14}  {"RIL":>6}  {"Original":>10}  {"CLAHE":>8}  {"Delta":>7}')
    print('  ' + '-' * 55)
    for seq in SEQS:
        r = clahe_results.get(seq, {})
        print(f'  {seq:<14}  {SEQ_RIL[seq]:>5.2f}%  '
              f'{r.get("original_mAP50",0):>9.1f}%  '
              f'{r.get("clahe_mAP50",0):>7.1f}%  '
              f'{r.get("delta",0):>+6.1f}%')
    orig_mean  = np.mean([clahe_results[s]['original_mAP50'] for s in SEQS])
    clahe_mean = np.mean([clahe_results[s]['clahe_mAP50']    for s in SEQS])
    print(f'  {"MEAN":<14}  {"":>6}  {orig_mean:>9.1f}%  '
          f'{clahe_mean:>7.1f}%  {clahe_mean-orig_mean:>+6.1f}%')

# ── Pedestrian size stratification summary ────────────────────────────────────
if ped_results:
    print()
    print('=' * 75)
    print('Pedestrian Size Stratification — YOLOv8x (Section V.D)')
    print('=' * 75)
    print(f'  {"Size Bin":<10}  {"Sequence":<14}  {"GT Count":>9}  {"mAP@50":>8}')
    print('  ' + '-' * 50)
    for bin_name, *_ in SIZE_BINS:
        seq_maps = []
        for seq in SEQS:
            d  = ped_results.get(bin_name, {}).get(seq, {})
            m  = d.get('mAP50', None)
            gt = d.get('gt_count', 0)
            if m is not None:
                print(f'  {bin_name:<10}  {seq:<14}  {gt:>9}  {m:>7.1f}%')
                seq_maps.append(m)
            else:
                print(f'  {bin_name:<10}  {seq:<14}  {gt:>9}  {"—":>8}')
        if seq_maps:
            print(f'  {bin_name:<10}  {"MEAN":<14}  {"":>9}  {np.mean(seq_maps):>7.1f}%')
        print()

TABLE IV — DETECTION EVALUATION
Zero-shot COCO pretrained weights | conf=0.25 | iou=0.50 | min_h=50px

mAP@50
  Model              Arch         sg0     sg1     sg2     sg3    Mean
  -----------------------------------------------------------------
  YOLOv8x            CNN        51.5%   47.0%   31.2%   41.1%   42.7%
  YOLO11x            CNN        53.0%   49.9%   30.5%   39.9%   43.3%
  RT-DETR-L (640)    Trans.     55.5%   53.3%   32.8%   47.7%   47.3%
  RT-DETR-X (640)    Trans.     56.7%   55.2%   34.4%   46.0%   48.1%
  RT-DETR-L          Trans.      6.5%    6.0%    3.4%    3.2%    4.8%
  RT-DETR-X          Trans.      0.3%    0.4%    0.9%    0.4%    0.5%

mAP@50:95
  Model              Arch         sg0     sg1     sg2     sg3    Mean
  -----------------------------------------------------------------
  YOLOv8x            CNN        36.6%   33.7%   25.0%   31.0%   31.6%
  YOLO11x            CNN        38.1%   36.1%   24.6%   29.8%   32.1%
  RT-DETR-L (640)    Trans.     41.3%   38.

## Cell 13 — Save All Results to Drive

In [12]:
import json, os, numpy as np

# Full JSON (all raw numbers)
json_path = f'{DRIVE_OUT}/detection_benchmark_perseq.json'
with open(json_path, 'w') as f:
    json.dump({
        'main_benchmark' : all_results,
        'clahe_ablation' : clahe_results,
        'ped_stratification': ped_results,
    }, f, indent=2)
print(f'✅ JSON saved: {json_path}')

# Human-readable text summary
txt_path = f'{DRIVE_OUT}/detection_benchmark_perseq.txt'
with open(txt_path, 'w') as f:
    f.write('=' * 80 + '\n')
    f.write('  SOLARDRIVE — DETECTION BENCHMARK RESULTS\n')
    f.write(f'  conf={CONF_THRESH}  iou={IOU_THRESH}  min_h={MIN_HEIGHT}px  imgsz={IMGSZ}\n')
    f.write('=' * 80 + '\n\n')

    for metric_key, metric_label in [('mAP50', 'mAP@50'), ('mAP50-95', 'mAP@50:95')]:
        f.write(f'{metric_label}\n')
        f.write(f'  {"Model":<18} {"Arch":<8}  s0      s1      s2      s3     Mean\n')
        f.write('-' * 72 + '\n')
        table_models_flat = [
            ('YOLOv8x', 'CNN'), ('YOLO11x', 'CNN'),
            ('RT-DETR-L (640)', 'Trans.'), ('RT-DETR-X (640)', 'Trans.'),
            ('RT-DETR-L', 'Trans.'), ('RT-DETR-X', 'Trans.'),
        ]
        for model_label, arch in table_models_flat:
            if model_label not in all_results:
                continue
            vals = []
            line = f'  {model_label:<18} {arch:<8}'
            for seq in SEQS:
                v = all_results[model_label].get(seq, {}).get('overall', {}).get(metric_key, 0)
                line += f'  {v:>5.1f}%'
                vals.append(v)
            line += f'  {np.mean(vals):>5.1f}%\n'
            f.write(line)
        f.write('\n')

    if clahe_results:
        f.write('CLAHE Ablation (YOLOv8x)\n')
        f.write(f'  {"Sequence":<14}  {"Original":>10}  {"CLAHE":>8}  {"Delta":>7}\n')
        f.write('-' * 48 + '\n')
        for seq in SEQS:
            r = clahe_results.get(seq, {})
            f.write(f'  {seq:<14}  {r.get("original_mAP50",0):>9.1f}%  '
                    f'{r.get("clahe_mAP50",0):>7.1f}%  {r.get("delta",0):>+6.1f}%\n')
        f.write('\n')

    if ped_results:
        f.write('Pedestrian Size Stratification (YOLOv8x)\n')
        f.write(f'  {"Size Bin":<10}  {"Sequence":<14}  {"GT":>5}  {"mAP@50":>8}\n')
        f.write('-' * 48 + '\n')
        for bin_name, *_ in SIZE_BINS:
            for seq in SEQS:
                d  = ped_results.get(bin_name, {}).get(seq, {})
                m  = d.get('mAP50', None)
                gt = d.get('gt_count', 0)
                m_str = f'{m:>7.1f}%' if m is not None else f'{"—":>8}'
                f.write(f'  {bin_name:<10}  {seq:<14}  {gt:>5}  {m_str}\n')
            f.write('\n')

print(f'✅ Text summary saved: {txt_path}')

✅ JSON saved: /content/drive/MyDrive/detection_benchmark_perseq.json
✅ Text summary saved: /content/drive/MyDrive/detection_benchmark_perseq.txt
